In [2]:
import re
import os
import glob
import pandas as pd

In [3]:
TRANSCRIPT_PATH = "dca_d1_1.txt"

In [4]:
with open(TRANSCRIPT_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print(raw_text[:3000])

((TAPE-HEADER "DCA TAPE 2; WASHINGTON NATIONAL ATCT; DEPARTURE RADAR CONTROLLER (DR1); 118.95 MHZ; MAY 26, 1992, 1524 EDT; TRANSCRIBER JLO"))

((COMMENT "POSITION WAS APPARENTLY COMBINED WITH DR1 HIGH; SEVERAL AIRCRAFT TRANSMITTING ON ANOTHER FREQUENCY; POSITION WAS LATER DECOMBINED; CONTROLLER CHANGES; POSITION WORKED DEPARTURES OFF ANDREWS AFB, AND DAVISON ARMY AIRFIELD AS WELL AS NATIONAL; DUE TO PROXIMITY OF DULLES AND BALTIMORE, MOST DEPARTURES WERE TRANSFERRED TO ONE OF THESE FACILITIES RATHER THAN BEING CHANGED TO THE CENTER AS IN PREVIOUS FACILITIES"))


((FROM DR1-1)
(TO DAL209)
(TEXT   DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGHT ZERO)
(TIMES    63.04    66.01))
  

((FROM DAL209)
(TO DR1-1)
(TEXT   LEFT TO TWO EIGHTY DELTA TWO OH NINE)
(TIMES    66.65    69.08))

((FROM AAL1581)
(TO DR1-1)
(TEXT   APPROACH AMERICAN EIGHT AH FIFTEEN EIGHTY ONE IS WITH YOU OUT OF TWO)
(TIMES    93.55    96.11))

((FROM DR1-1)
(TO AAL1581)
(TEXT   AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPAR

In [5]:
message_pattern = re.compile(
    r"\(\(FROM\s+(.*?)\)\s*"
    r"\(TO\s+(.*?)\)\s*"
    r"\(TEXT\s+(.*?)\)\s*"
    r"\(TIMES\s+([0-9.]+)\s+([0-9.]+)\)",
    re.DOTALL
)

matches = message_pattern.findall(raw_text)
len(matches)

397

In [6]:
rows = []
for frm, to, text, t1, t2 in matches:
    rows.append({
        "from": frm.strip(),
        "to": to.strip(),
        "text": text.strip(),
        "start_time": float(t1),
        "end_time": float(t2),
        "duration_s": float(t2) - float(t1)
    })

df_msgs = pd.DataFrame(rows)
df_msgs.head(20)

,from,to,text,start_time,end_time,duration_s
0,DR1-1,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,63.04,66.01,2.97
1,DAL209,DR1-1,LEFT TO TWO EIGHTY DELTA TWO OH NINE,66.65,69.08,2.43
2,AAL1581,DR1-1,APPROACH AMERICAN EIGHT AH FIFTEEN EIGHTY ONE ...,93.55,96.11,2.56
3,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,96.63,101.06,4.43
4,AAL1581,DR1-1,UP TO ONE SEVEN THOUSAND AMERICAN FIFTEEN EIGH...,101.30,103.46,2.16
5,DR1-1,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,106.51,109.48,2.97
6,DAL209,DR1-1,LEFT TO TWO FORTY DELTA TWO OH NINE,109.85,112.06,2.21
7,DR1-1,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,195.95,201.46,5.51
8,DAL209,DR1-1,OKAY WE'LL GO DIRECT LINDEN RIGHT NOW DELTA TW...,201.98,204.77,2.79
9,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,205.16,208.22,3.06


In [7]:
def clean_atc_text(text):
    text = text.upper()
    text = re.sub(r"\(UNINTELLIGIBLE.*?\)", " UNINTELLIGIBLE ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_msgs["clean_text"] = df_msgs["text"].apply(clean_atc_text)
df_msgs[["from", "to", "clean_text"]].head(20)

,from,to,clean_text
0,DR1-1,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
1,DAL209,DR1-1,LEFT TO TWO EIGHTY DELTA TWO OH NINE
2,AAL1581,DR1-1,APPROACH AMERICAN EIGHT AH FIFTEEN EIGHTY ONE ...
3,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
4,AAL1581,DR1-1,UP TO ONE SEVEN THOUSAND AMERICAN FIFTEEN EIGH...
5,DR1-1,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
6,DAL209,DR1-1,LEFT TO TWO FORTY DELTA TWO OH NINE
7,DR1-1,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
8,DAL209,DR1-1,OKAY WE'LL GO DIRECT LINDEN RIGHT NOW DELTA TW...
9,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...


In [8]:
df_ctrl = df_msgs[df_msgs["from"].str.startswith("DR", na=False)].copy()
df_ctrl[["from", "to", "clean_text"]].head(20)

,from,to,clean_text
0,DR1-1,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
3,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
5,DR1-1,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
7,DR1-1,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
9,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
11,DR1-1,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...
13,DR1-1,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
16,DR1-1,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...
18,DR1-1,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...
19,DR1-1,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...


In [9]:
ENTITY_TYPES = [
    "CALLSIGN",
    "ACTION",
    "RUNWAY",
    "RUNWAY_HEADING",
    "HEADING",
    "ALTITUDE",
    "FLIGHT_LEVEL",
    "WAYPOINT",
    "FREQUENCY",
    "SPEED",
    "DIRECTION",
    "NAV_INSTRUCTION"
]

ENTITY_TYPES

['CALLSIGN',
 'ACTION',
 'RUNWAY',
 'RUNWAY_HEADING',
 'HEADING',
 'ALTITUDE',
 'FLIGHT_LEVEL',
 'WAYPOINT',
 'FREQUENCY',
 'SPEED',
 'DIRECTION',
 'NAV_INSTRUCTION']

In [10]:
patterns = {
    "HEADING": r"\bHEADING\s+((?:ZERO|ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NINER|NINE|OH|\s)+)\b",
    "ALTITUDE": r"\b(?:MAINTAIN|CLIMB AND MAINTAIN|DESCEND AND MAINTAIN)\s+((?:FLIGHT LEVEL\s+)?(?:ZERO|ONE|TWO|THREE|FOUR|FIVE|SIX|SEVEN|EIGHT|NINER|NINE|OH|\s)+)\b",
    "RUNWAY": r"\bRUNWAY\s+([0-9]{1,2}[LRC]?)\b",
    "WAYPOINT": r"\bDIRECT\s+([A-Z0-9]{3,10})\b",
    "FREQUENCY": r"\bONE\s+TWO[0-9A-Z\s]*POINT\s+[0-9A-Z\s]+\b|\b[0-9]{3}\.[0-9]{1,2}\b",
    "DIRECTION": r"\b(LEFT|RIGHT)\b"
}

action_keywords = [
    "TURN LEFT",
    "TURN RIGHT",
    "FLY HEADING",
    "CLIMB AND MAINTAIN",
    "DESCEND AND MAINTAIN",
    "MAINTAIN",
    "PROCEED DIRECT",
    "CONTACT",
    "RESUME YOUR OWN NAVIGATION",
    "JOIN",
    "EXPECT",
    "RADAR CONTACT",
    "DO NOT EXCEED",
    "EXPEDITE"
]

In [11]:
def weak_extract_entities(text):
    entities = {
        "ACTION": [],
        "HEADING": [],
        "ALTITUDE": [],
        "RUNWAY": [],
        "WAYPOINT": [],
        "FREQUENCY": [],
        "DIRECTION": []
    }

    for action in action_keywords:
        if action in text:
            entities["ACTION"].append(action)

    for label, pattern in patterns.items():
        matches = re.findall(pattern, text)
        if matches:
            entities[label] = matches if isinstance(matches, list) else [matches]

    return entities

df_ctrl["weak_entities"] = df_ctrl["clean_text"].apply(weak_extract_entities)
df_ctrl[["clean_text", "weak_entities"]].head(20)

,clean_text,weak_entities
0,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO EIG..."
3,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,"{'ACTION': ['CLIMB AND MAINTAIN', 'MAINTAIN', ..."
5,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO FOU..."
7,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,"{'ACTION': ['TURN RIGHT', 'RESUME YOUR OWN NAV..."
9,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO NIN..."
11,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,"{'ACTION': ['CONTACT'], 'HEADING': [], 'ALTITU..."
13,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,"{'ACTION': ['TURN LEFT'], 'HEADING': ['TWO THR..."
16,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,"{'ACTION': ['FLY HEADING', 'CLIMB AND MAINTAIN..."
18,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,"{'ACTION': ['CLIMB AND MAINTAIN', 'MAINTAIN'],..."
19,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,"{'ACTION': ['CLIMB AND MAINTAIN', 'MAINTAIN'],..."


In [12]:
df_ctrl["actions"] = df_ctrl["weak_entities"].apply(lambda x: x["ACTION"])
df_ctrl["headings"] = df_ctrl["weak_entities"].apply(lambda x: x["HEADING"])
df_ctrl["altitudes"] = df_ctrl["weak_entities"].apply(lambda x: x["ALTITUDE"])
df_ctrl["runways"] = df_ctrl["weak_entities"].apply(lambda x: x["RUNWAY"])
df_ctrl["waypoints"] = df_ctrl["weak_entities"].apply(lambda x: x["WAYPOINT"])
df_ctrl["frequencies"] = df_ctrl["weak_entities"].apply(lambda x: x["FREQUENCY"])
df_ctrl["directions"] = df_ctrl["weak_entities"].apply(lambda x: x["DIRECTION"])

df_ctrl[[
    "clean_text",
    "actions",
    "headings",
    "altitudes",
    "runways",
    "waypoints",
    "frequencies",
    "directions"
]].head(30)

,clean_text,actions,headings,altitudes,runways,waypoints,frequencies,directions
0,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,[TURN LEFT],[TWO EIGHT ZERO],[],[],[],[],[LEFT]
3,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,"[CLIMB AND MAINTAIN, MAINTAIN, CONTACT, RADAR ...",[],[ONE SEVEN ],[],[],[],[]
5,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,[TURN LEFT],[TWO FOUR ZERO],[],[],[],[],[LEFT]
7,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,"[TURN RIGHT, RESUME YOUR OWN NAVIGATION]",[TWO SEVEN ZERO ],[],[],[LINDEN],[],[RIGHT]
9,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO NINER ZERO ],[],[],[],[],[LEFT]
11,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,[CONTACT],[],[],[],[],[],[]
13,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO THREE ZERO ],[],[],[],[],[LEFT]
16,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,"[FLY HEADING, CLIMB AND MAINTAIN, MAINTAIN]",[THREE THREE ZERO ],[FOUR ],[],[],[],[]
18,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[FLIGHT LEVEL TWO THREE ZERO],[],[],[],[]
19,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[],[],[],[],[]


In [13]:
df_ctrl["callsign"] = df_ctrl["to"].str.strip().str.upper()
df_ctrl[["callsign", "clean_text"]].head(20)

,callsign,clean_text
0,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
3,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
5,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
7,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
9,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
11,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...
13,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
16,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...
18,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...
19,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...


In [14]:
training_df = df_ctrl[[
    "callsign",
    "clean_text",
    "actions",
    "headings",
    "altitudes",
    "runways",
    "waypoints",
    "frequencies",
    "directions"
]].copy()

training_df.head(30)

,callsign,clean_text,actions,headings,altitudes,runways,waypoints,frequencies,directions
0,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,[TURN LEFT],[TWO EIGHT ZERO],[],[],[],[],[LEFT]
3,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,"[CLIMB AND MAINTAIN, MAINTAIN, CONTACT, RADAR ...",[],[ONE SEVEN ],[],[],[],[]
5,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,[TURN LEFT],[TWO FOUR ZERO],[],[],[],[],[LEFT]
7,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,"[TURN RIGHT, RESUME YOUR OWN NAVIGATION]",[TWO SEVEN ZERO ],[],[],[LINDEN],[],[RIGHT]
9,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO NINER ZERO ],[],[],[],[],[LEFT]
11,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,[CONTACT],[],[],[],[],[],[]
13,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,[TURN LEFT],[TWO THREE ZERO ],[],[],[],[],[LEFT]
16,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,"[FLY HEADING, CLIMB AND MAINTAIN, MAINTAIN]",[THREE THREE ZERO ],[FOUR ],[],[],[],[]
18,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[FLIGHT LEVEL TWO THREE ZERO],[],[],[],[]
19,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,"[CLIMB AND MAINTAIN, MAINTAIN]",[],[],[],[],[],[]


In [15]:
number_words = {
    "ZERO": "0",
    "OH": "0",
    "ONE": "1",
    "TWO": "2",
    "THREE": "3",
    "FOUR": "4",
    "FIVE": "5",
    "SIX": "6",
    "SEVEN": "7",
    "EIGHT": "8",
    "NINE": "9",
    "NINER": "9"
}

In [16]:
def spoken_digits_to_string(text):
    words = text.upper().strip().split()
    digits = []

    for w in words:
        if w in number_words:
            digits.append(number_words[w])

    return "".join(digits)

In [17]:
def normalize_heading_spoken(text):
    digit_str = spoken_digits_to_string(text)
    if digit_str == "":
        return None
    return int(digit_str)

In [18]:
def normalize_altitude_spoken(text):
    text = text.upper().strip()

    if text.startswith("FLIGHT LEVEL"):
        remainder = text.replace("FLIGHT LEVEL", "").strip()
        digit_str = spoken_digits_to_string(remainder)
        if digit_str == "":
            return None
        return int(digit_str) * 100

    if "THOUSAND" in text:
        prefix = text.replace("THOUSAND", "").strip()
        digit_str = spoken_digits_to_string(prefix)
        if digit_str == "":
            return None
        return int(digit_str) * 1000

    digit_str = spoken_digits_to_string(text)
    if digit_str == "":
        return None
    return int(digit_str)

In [19]:
def normalize_frequency_spoken(text):
    text = text.upper().strip()
    parts = text.split("POINT")

    if len(parts) == 1:
        left = spoken_digits_to_string(parts[0])
        return float(left) if left != "" else None

    left = spoken_digits_to_string(parts[0])
    right = spoken_digits_to_string(parts[1])

    if left == "" or right == "":
        return None

    return float(f"{left}.{right}")

In [20]:
print("Heading:", normalize_heading_spoken("TWO EIGHT ZERO"))
print("Heading:", normalize_heading_spoken("ZERO NINER ZERO"))

print("Altitude:", normalize_altitude_spoken("ONE SEVEN THOUSAND"))
print("Altitude:", normalize_altitude_spoken("THREE THOUSAND"))
print("Altitude:", normalize_altitude_spoken("FLIGHT LEVEL TWO THREE ZERO"))

print("Frequency:", normalize_frequency_spoken("ONE TWO FOUR POINT EIGHT"))
print("Frequency:", normalize_frequency_spoken("ONE TWO SIX POINT THREE FIVE"))

Heading: 280
Heading: 90
Altitude: 17000
Altitude: 3000
Altitude: 23000
Frequency: 124.8
Frequency: 126.35


In [23]:
def first_or_none(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return None

df_ctrl["heading_raw"] = df_ctrl["headings"].apply(first_or_none)
df_ctrl["altitude_raw"] = df_ctrl["altitudes"].apply(first_or_none)
df_ctrl["frequency_raw"] = df_ctrl["frequencies"].apply(first_or_none)

In [24]:
df_ctrl["heading_numeric"] = df_ctrl["heading_raw"].apply(
    lambda x: normalize_heading_spoken(x) if isinstance(x, str) else None
)

df_ctrl["altitude_numeric_ft"] = df_ctrl["altitude_raw"].apply(
    lambda x: normalize_altitude_spoken(x) if isinstance(x, str) else None
)

df_ctrl["frequency_numeric"] = df_ctrl["frequency_raw"].apply(
    lambda x: normalize_frequency_spoken(x) if isinstance(x, str) and "POINT" in x.upper() else None
)

In [25]:
df_ctrl[[
    "clean_text",
    "heading_raw",
    "heading_numeric",
    "altitude_raw",
    "altitude_numeric_ft",
    "frequency_raw",
    "frequency_numeric"
]].head(30)

,clean_text,heading_raw,heading_numeric,altitude_raw,altitude_numeric_ft,frequency_raw,frequency_numeric
0,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...,TWO EIGHT ZERO,280.0,None,NaN,None,NaN
3,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...,None,NaN,ONE SEVEN,17.0,None,NaN
5,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO,TWO FOUR ZERO,240.0,None,NaN,None,NaN
7,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...,TWO SEVEN ZERO,270.0,None,NaN,None,NaN
9,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,TWO NINER ZERO,290.0,None,NaN,None,NaN
11,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...,None,NaN,None,NaN,None,NaN
13,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...,TWO THREE ZERO,230.0,None,NaN,None,NaN
16,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...,THREE THREE ZERO,330.0,FOUR,4.0,None,NaN
18,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...,None,NaN,FLIGHT LEVEL TWO THREE ZERO,23000.0,None,NaN
19,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...,None,NaN,None,NaN,None,NaN


In [26]:
airline_map = {
    "DELTA": "DAL",
    "AMERICAN": "AAL",
    "UNITED": "UAL",
    "U S AIR": "USA",
    "US AIR": "USA",
    "NORTHWEST": "NWA",
    "T W A": "TWA",
    "CACTUS": "AWE",
    "AIR WISCONSIN": "AWI",
    "SWIFT": "SWIFT",
    "TRADOC": "TRADOC",
    "PACER": "PACER",
    "SABRE": "SABRE",
    "SABRE LINER": "SABRE",
    "KING AIR": "KINGAIR",
    "HAWKER": "HAWKER",
    "AERO STAR": "AEROSTAR",
    "NAVY": "NAVY",
    "ARMY": "ARMY",
    "HENSON": "HNA"
}

In [27]:
letter_words = {
    "ALFA": "A", "ALPHA": "A",
    "BRAVO": "B",
    "CHARLIE": "C",
    "DELTA": "D",
    "ECHO": "E",
    "FOXTROT": "F", "FOX": "F",
    "GOLF": "G",
    "HOTEL": "H",
    "INDIA": "I",
    "JULIET": "J",
    "KILO": "K",
    "LIMA": "L",
    "MIKE": "M",
    "NOVEMBER": "N",
    "OSCAR": "O",
    "PAPA": "P",
    "QUEBEC": "Q",
    "ROMEO": "R",
    "SIERRA": "S",
    "TANGO": "T",
    "UNIFORM": "U",
    "VICTOR": "V",
    "WHISKEY": "W",
    "XRAY": "X", "X-RAY": "X",
    "YANKEE": "Y",
    "ZULU": "Z"
}

In [28]:
def spoken_alphanum_to_string(text):
    words = text.upper().strip().split()
    out = []

    for w in words:
        if w in number_words:
            out.append(number_words[w])
        elif w in letter_words:
            out.append(letter_words[w])

    return "".join(out)

In [42]:
def normalize_callsign_from_metadata_or_text(metadata_callsign, spoken_text=None):
    metadata_callsign = None if pd.isna(metadata_callsign) else str(metadata_callsign).strip().upper()
    spoken_text = None if pd.isna(spoken_text) else str(spoken_text).strip().upper()

    # If transcript metadata already gives a clean aircraft ID, trust it
    if metadata_callsign:
        return metadata_callsign

    # Otherwise try to normalize from spoken text
    if spoken_text:
        spoken_prefix = extract_spoken_callsign_prefix(spoken_text)
        if spoken_prefix:
            return normalize_spoken_callsign(spoken_prefix)

    return None

In [43]:
small_number_words = {
    "ZERO": 0, "OH": 0,
    "ONE": 1, "TWO": 2, "THREE": 3, "FOUR": 4, "FIVE": 5,
    "SIX": 6, "SEVEN": 7, "EIGHT": 8, "NINE": 9, "NINER": 9,
    "TEN": 10, "ELEVEN": 11, "TWELVE": 12, "THIRTEEN": 13,
    "FOURTEEN": 14, "FIFTEEN": 15, "SIXTEEN": 16, "SEVENTEEN": 17,
    "EIGHTEEN": 18, "NINETEEN": 19,
    "TWENTY": 20, "THIRTY": 30, "FORTY": 40, "FIFTY": 50,
    "SIXTY": 60, "SEVENTY": 70, "EIGHTY": 80, "NINETY": 90
}

In [44]:
def parse_spoken_number_chunk(chunk_words):
    total = 0
    i = 0
    words = [w.upper() for w in chunk_words]

    while i < len(words):
        w = words[i]

        if w in ["TWENTY", "THIRTY", "FORTY", "FIFTY", "SIXTY", "SEVENTY", "EIGHTY", "NINETY"]:
            val = small_number_words[w]
            if i + 1 < len(words) and words[i + 1] in number_words:
                val += int(number_words[words[i + 1]])
                i += 1
            total += val
        elif w in small_number_words:
            total += small_number_words[w]
        elif w in number_words:
            total += int(number_words[w])

        i += 1

    return str(total)

In [45]:
def spoken_callsign_suffix_to_string(text):
    words = text.upper().strip().split()
    out = []
    chunk = []

    for w in words:
        if w in letter_words:
            if chunk:
                out.append(parse_spoken_number_chunk(chunk))
                chunk = []
            out.append(letter_words[w])
        elif w in small_number_words or w in number_words:
            chunk.append(w)

    if chunk:
        out.append(parse_spoken_number_chunk(chunk))

    return "".join(out)

In [46]:
def is_tail_number_style(callsign):
    if callsign is None:
        return False

    s = str(callsign).strip().upper()

    if re.fullmatch(r"N[0-9A-Z]+", s):
        return True

    return False

In [47]:
def extract_spoken_callsign_prefix(text):
    text = str(text).strip().upper()

    prefixes = [
        "DELTA",
        "AMERICAN",
        "UNITED",
        "U S AIR",
        "US AIR",
        "NORTHWEST",
        "T W A",
        "CACTUS",
        "AIR WISCONSIN",
        "SWIFT",
        "TRADOC",
        "PACER",
        "SABRE LINER",
        "SABRE",
        "KING AIR",
        "HAWKER",
        "AERO STAR",
        "NAVY",
        "ARMY",
        "HENSON",
        "NOVEMBER"
    ]

    prefixes = sorted(prefixes, key=len, reverse=True)

    for prefix in prefixes:
        if text.startswith(prefix):
            words = text.split()

            # capture the first few words as the spoken callsign phrase
            # enough to include number/letter suffixes like "SABRE ONE SIX ZERO WHISKEY"
            max_len = min(len(words), 6)
            return " ".join(words[:max_len])

    return None

In [48]:
df_ctrl["callsign_metadata"] = df_ctrl["to"].astype(str).str.upper().str.strip()
df_ctrl["callsign_spoken_text"] = df_ctrl["clean_text"].apply(extract_spoken_callsign_prefix)

df_ctrl["callsign_final"] = df_ctrl.apply(
    lambda row: normalize_callsign_from_metadata_or_text(
        row["callsign_metadata"],
        row["callsign_spoken_text"]
    ),
    axis=1
)

In [51]:
training_df = df_ctrl[[
    "callsign_final",
    "callsign_metadata",
    "callsign_spoken_text",
    "clean_text",
    "actions",
    "headings",
    "heading_numeric",
    "altitudes",
    "altitude_numeric_ft",
    "runways",
    "waypoints",
    "frequencies",
    "frequency_numeric",
    "directions"
]].copy()

In [53]:
df_ctrl[[
    "callsign_metadata",
    "callsign_spoken_text",
    "callsign_final",
    "clean_text"
]].head(15)

,callsign_metadata,callsign_spoken_text,callsign_final,clean_text
0,DAL209,DELTA TWO ZERO NINE TURN LEFT,DAL209,DELTA TWO ZERO NINE TURN LEFT HEADING TWO EIGH...
3,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTURE,AAL1581,AMERICAN FIFTEEN EIGHTY ONE WASHINGTON DEPARTU...
5,DAL209,DELTA TWO OH NINE TURN LEFT,DAL209,DELTA TWO OH NINE TURN LEFT HEADING TWO FOUR ZERO
7,DAL209,DELTA TWO OH NINE TURN RIGHT,DAL209,DELTA TWO OH NINE TURN RIGHT AH HEADING TWO SE...
9,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
11,DAL209,DELTA TWO ZERO NINE CONTACT DULLES,DAL209,DELTA TWO ZERO NINE CONTACT DULLES APPROACH CO...
13,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT,AAL1581,AMERICAN FIFTEEN EIGHTY ONE TURN LEFT HEADING ...
16,N99G,AERO STAR NINE NINE GOLF FLY,N99G,AERO STAR NINE NINE GOLF FLY HEADING THREE THR...
18,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE,DAL743,DELTA SEVEN FORTY THREE WASHINGTON DEPARTURE C...
19,N99G,AERO STAR NINE NINE GOLF CLIMB,N99G,AERO STAR NINE NINE GOLF CLIMB AND MAINTAIN AH...
